In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie_raw_object_id.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia.head()


In [ ]:
data_gdynia_long = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

1.

In [ ]:
mean_num_storeys_street_agg = data_gdynia.groupby("addr_street")[
    "number_of_storeys"
].agg("mean")
mean_num_storeys_street_trans = data_gdynia.groupby("addr_street")[
    "number_of_storeys"
].transform("mean")

In [ ]:
mean_num_storeys_street_agg.head()

In [ ]:
mean_num_storeys_street_trans

In [ ]:
assert len(mean_num_storeys_street_trans) == len(data_gdynia)
assert mean_num_storeys_street_trans.isna().equals(data_gdynia["addr_street"].isna())
assert mean_num_storeys_street_trans.index.equals(data_gdynia.index)

2.

In [ ]:
data_gdynia = data_gdynia.assign(
    street_mean=lambda d: d.groupby("addr_street")["number_of_storeys"].transform(
        "mean"
    ),
    storeys_vs_street=lambda d: d["number_of_storeys"] - d["street_mean"],
)

In [ ]:
data_gdynia.loc[
    data_gdynia["storeys_vs_street"].abs().nlargest(10).index,
    ["addr_street", "number_of_storeys", "street_mean", "storeys_vs_street"],
]

In [ ]:
assert np.isclose(
    data_gdynia.groupby("addr_street")["storeys_vs_street"].agg("mean"), 0
).all()

3.

In [ ]:
style_share_per_street = (
    data_gdynia.groupby(["addr_street", "arch_style"])
    .agg(num_build=("osm_id", "size"))
    .assign(
        street_total=lambda d: d.groupby("addr_street")["num_build"].transform("sum"),
        share=lambda d: d["num_build"] / d["street_total"],
    )
)

In [ ]:
style_share_per_street.head(10)

In [ ]:
sum_shares_per_street = style_share_per_street.groupby("addr_street").agg(
    sum_shares=("share", "sum")
)
assert np.isclose(sum_shares_per_street, 1).all()

5.

In [ ]:
data_gdynia = data_gdynia.assign(
    mean_street=lambda d: d.groupby("addr_street")["number_of_storeys"].transform(
        "mean"
    ),
    std_street=lambda d: d.groupby("addr_street")["number_of_storeys"].transform(
        np.std
    ),
    z_in_street=lambda d: (d["number_of_storeys"] - d["mean_street"]) / d["std_street"],
)

In [ ]:
data_gdynia.loc[
    data_gdynia["z_in_street"].abs() > 2,
    ["number_of_storeys", "mean_street", "std_street", "z_in_street"],
]

In [ ]:
assert np.isclose(data_gdynia.groupby("addr_street")["z_in_street"].agg("sum"), 0).all()

6.

In [ ]:
data_gdynia_rank = data_gdynia.assign(
    rank_min=lambda d: d.groupby("addr_street")["number_of_storeys"].rank(
        ascending=False, method="min"
    ),
    rank_dense=lambda d: d.groupby("addr_street")["number_of_storeys"].rank(
        ascending=False, method="dense"
    ),
    rank_first=lambda d: d.groupby("addr_street")["number_of_storeys"].rank(
        ascending=False, method="first"
    ),
)

data_gdynia_rank.loc[
    :, ["addr_street", "number_of_storeys", "rank_min", "rank_dense", "rank_first"]
]

In [ ]:
assert (
    data_gdynia_rank.groupby("addr_street")["rank_first"].max()
    == data_gdynia_rank.groupby("addr_street")["rank_first"].size()
).all()

7.

In [ ]:
data_gdynia = data_gdynia.assign(
    order_in_street=lambda d: (
        d.sort_values(["date_of_checking", "id"]).groupby("addr_street").cumcount() + 1
    )
)

In [ ]:
data_gdynia.loc[:, ["addr_street", "date_of_checking", "order_in_street"]]

In [ ]:
assert (
    data_gdynia.groupby("addr_street")["order_in_street"].max()
    == data_gdynia.groupby("addr_street")["osm_id"].agg("size")
).all()

8.

In [ ]:
data_gdynia_top_2_height = (
    data_gdynia.sort_values(["number_of_storeys", "id"], ascending=[False, True])
    .groupby("addr_street")[["addr_street", "number_of_storeys"]]
    .head(2)
    .sort_values(["addr_street", "number_of_storeys"], ascending=[True, False])
)
data_gdynia_top_2_height.head(20)

In [ ]:
assert (data_gdynia_top_2_height.groupby("addr_street").size() <= 2).all()

9.

In [ ]:
data_gdynia_most_frequent_style = data_gdynia.groupby(
    ["addr_street", "arch_style"]
).agg(num_build=("osm_id", "size"))

In [ ]:
data_gdynia_most_frequent_style2 = (
    data_gdynia_most_frequent_style.sort_values(
        ["num_build", "arch_style"], ascending=[False, True]
    )
    .groupby("addr_street")
    .head(2)
    .sort_values(
        ["addr_street", "num_build", "arch_style"], ascending=[True, False, True]
    )
)
data_gdynia_most_frequent_style2.head(20)

In [ ]:
assert (data_gdynia_most_frequent_style2.groupby("addr_street").size() <= 2).all()

In [ ]:
data_gdynia_most_frequent_style["rank"] = data_gdynia_most_frequent_style.groupby(
    "addr_street"
)["num_build"].rank(method="dense", ascending=False)

In [ ]:
mask = data_gdynia_most_frequent_style["rank"] <= 2
data_gdynia_most_frequent_style_rank = data_gdynia_most_frequent_style.loc[
    mask
].sort_values(["addr_street", "rank"], ascending=[True, True])

In [ ]:
assert data_gdynia_most_frequent_style_rank["rank"].nunique() == 2
assert (data_gdynia_most_frequent_style_rank["rank"] <= 2).all()

In [ ]:
data_gdynia_most_frequent_style_rank.loc[
    (data_gdynia_most_frequent_style_rank.groupby("addr_street").transform("size") > 2)
]

10.

In [ ]:
data_gdynia_sum_build_style = data_gdynia.groupby("arch_style").agg(
    num_style=("osm_id", "size")
)
data_gdynia_sum_build_style

In [ ]:
assert len(data_gdynia_sum_build_style) == data_gdynia["arch_style"].nunique()

In [ ]:
data_gdynia["build_street_same_style"] = data_gdynia.groupby(
    ["addr_street", "arch_style"]
).transform("size")

In [ ]:
data_gdynia.loc[:, ["addr_street", "arch_style", "build_street_same_style"]]

In [ ]:
assert (
    data_gdynia["build_street_same_style"].isna().sum()
    == data_gdynia["addr_street"].isna().sum()
)

In [ ]:
data_gdynia.groupby("addr_street").transform("size")

In [ ]:
data_gdynia_streets_over_20 = data_gdynia.loc[
    data_gdynia.groupby(["addr_street"]).transform("size") >= 20
]
data_gdynia_streets_over_20.head()

In [ ]:
assert data_gdynia_streets_over_20["addr_street"].nunique() == 13
assert len(data_gdynia_streets_over_20) == 605